# Sofascore website capture v3

Explore one completed match by capturing JSON responses **the normal website itself loads**.
The current scraper and historical collection notebook are unchanged.

**Findings:** a fresh Selenium session returned 56 HTTP 403 responses. The same historical match subsequently loaded in the existing Chrome profile, including lineups and injuries. We can now import its already-loaded Network responses through HAR files. Full coverage and request savings still need a successful archive comparison. Details: `../sofascore_web_v3/README.md`.

This notebook defaults to a local review. The preferred route is **your existing Chrome → Network response export → offline import**: set `HAR_FILES` below. The optional Selenium experiment is separate. The importer makes no requests; the website itself still does, potentially in bursts.

## 1. Setup

Select the **Python (xDiyo)** kernel. Run from this repository; existing dependencies are Selenium, pandas, and pyarrow. The original collector must be stopped before starting this experiment because both use the shared request ledger lock.

In [1]:
from pathlib import Path
from dataclasses import replace
import json
import sys
import time

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "scraping/sofascore_web_v3/capture.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the xDiyo repository")
sys.dont_write_bytecode = True
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from scraping.sofascore_web_v3.capture import PROFILE, WebsiteCapture, reference_from_cache
from scraping.sofascore_web_v3.import_har import import_har
from scraping.sofascore.export import read_tables
print("Repository:", ROOT)

Repository: C:\Users\luisi\Documents\Programming\Python\xDiyo


## 2. Parameters

Keep `RUN_EXPERIMENT = False` when importing Network exports. Select a completed `EVENT_ID` and list the HAR files in `HAR_FILES`. Captured event metadata can supply the identity; the old cache is an optional fallback. Multiple files are merged chronologically, with duplicate observations removed.

For a challenge, set `MANUAL_CAPTCHA = True`: Chrome first opens the homepage for `CAPTCHA_SECONDS`, during which **you** complete the challenge. It then opens the selected match. The original access latch stays unchanged; any active shared cooldown is respected.

During `BROWSE_SECONDS`, select the visible match views you want captured. Both the warmup and match browsing consume the same observed request/time budget. Browser bursts can overshoot the request limit. The notebook cannot apply the old scraper's random delay between website-generated requests.

In [2]:
RUN_EXPERIMENT = False
EVENT_ID = 12499375  # Borussia M'gladbach vs Bayer Leverkusen, 2024/25 round 1
HAR_FILES = []  # Example: [Path(r"C:\Users\luisi\Downloads\sofascore-event-12499375.har")]

INCLUDE_LINEUPS = True
INCLUDE_INJURIES = True  # Requires the lineups response even if roster export is off.
INCLUDE_HEATMAPS = True
# xG/shotmap, statistics, pregame positions, goal sequences, and timing remain enabled.
# Commentary is never retained by this experiment.

MANUAL_CAPTCHA = False
CAPTCHA_SECONDS = 60
BROWSE_SECONDS = 60
MAX_API_REQUESTS = 80
MAX_SESSION_SECONDS = 180
DRIVER_PATH = None  # Optional matching chromedriver.exe; None uses Selenium Manager.

OUTPUT_ROOT = ROOT / "data/website_capture_v3"
LEDGER_ROOT = ROOT / "data/collection_state"
SAVED_RUN = OUTPUT_ROOT / "20260912T011212Z-160350af"

profile = replace(PROFILE, include_lineups=INCLUDE_LINEUPS,
                  include_injuries=INCLUDE_INJURIES, include_heatmaps=INCLUDE_HEATMAPS)
reference = None
if (LEDGER_ROOT / "manifest.sqlite").is_file():
    try:
        reference = reference_from_cache(EVENT_ID, LEDGER_ROOT)
    except ValueError:
        pass
if reference:
    event = reference["event"]
    print(f"Selected: {event['homeTeam']['name']} vs {event['awayTeam']['name']}")
    print("Event:", EVENT_ID, "Season:", event["season"]["year"])
else:
    print("The HAR must contain event metadata for", EVENT_ID)
if RUN_EXPERIMENT and (HAR_FILES or reference is None):
    raise ValueError("Use HAR import or Selenium separately; Selenium also needs cached fixture identity")
print("Live collection enabled:", RUN_EXPERIMENT)

Selected: Borussia M'gladbach vs Bayer 04 Leverkusen
Event: 12499375 Season: 24/25
Live collection enabled: False


## 3. Import responses from the working Chrome session

**Without an extension:** in the match tab, open DevTools → Network, browse the relevant views, and choose **Export HAR (sanitized)**. Set its path in `HAR_FILES`. Open DevTools before browsing to include subsequent requests. If it missed the initial load, one deliberate page reload may be needed; do not replay individual requests. The archive must include response bodies.

**With the optional local extension:** load `../sofascore_web_v3/devtools_extension` as an unpacked extension, reopen DevTools, and select the **xDiyo** panel. Set the event ID, click **Start recording**, then **Read existing Network log** to collect responses already listed. Keep DevTools open while browsing normally. **Export HAR** saves selected response bodies without cookies, headers, or commentary. After exporting, a new recording can target another match in the same browser session.

The extension is provided as source and has not been installed automatically. Its recording limits only bound local capture; they do not stop website requests. Missing bodies are reported as failures instead of silently treated as empty data. Importing neither opens Chrome nor modifies the old collector's ledger or prepared seasons.

In [3]:
run_report = None
if HAR_FILES:
    run_report = import_har(HAR_FILES, event_id=EVENT_ID, reference=reference,
                            output_root=OUTPUT_ROOT, profile=profile)
    print("Imported into:", run_report["directory"])
    print("Requested component coverage complete:", run_report["complete"])
else:
    print("No HAR files selected; no data imported.")

No HAR files selected; no data imported.


## 4. Optional Selenium experiment

Enable the parameter above, rerun the parameter cell, then run this cell. Keep the Chrome window visible and use its normal controls during each timed interval. A 403 during match browsing ends this attempt; it does not trigger an automatic retry. A switch to a different event also stops browsing.

The browser closes and partial captures are exported at the end. Interrupting the cell saves the captured subset when possible. Closing Chrome manually also leaves the raw checkpoint available. Each new run uses a new output directory; this does not resume or overwrite a historical-season run.

In [4]:
def watch(capture, seconds, label):
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline and not capture.stop_reason:
        capture.observe(min(10, max(0, deadline - time.monotonic())))
        state = capture.report()
        print(f"{label}: {state['observed_api_requests']} observed API requests; "
              f"status counts {state['http_status_counts']}; stop={state['stop_reason']}", flush=True)


if RUN_EXPERIMENT:
    capture = WebsiteCapture(reference, output_root=OUTPUT_ROOT, ledger_root=LEDGER_ROOT,
                             driver_path=DRIVER_PATH, profile=profile,
                             max_api_requests=MAX_API_REQUESTS, max_seconds=MAX_SESSION_SECONDS)
    try:
        capture.start()
        print("Experiment output:", capture.directory, flush=True)
        try:
            if MANUAL_CAPTCHA:
                print(f"Complete any challenge manually in Chrome within {CAPTCHA_SECONDS}s.", flush=True)
                capture.open_homepage()
                watch(capture, CAPTCHA_SECONDS, "Human challenge window")
            if not capture.stop_reason:
                capture.open_match()
                print(f"Browse the selected match's visible views for {BROWSE_SECONDS}s.", flush=True)
                watch(capture, BROWSE_SECONDS, "Match browsing")
            if not capture.stop_reason:
                capture.snapshot()
        except KeyboardInterrupt:
            print("Interrupted; exporting the captured subset.", flush=True)
        run_report = capture.finish()
    finally:
        capture.close()
else:
    print("Live collection skipped. The next cell reads the saved diagnostic, if present.")

Live collection skipped. The next cell reads the saved diagnostic, if present.


## 5. Coverage and request counts

`not_observed` means the required response was not captured. It does **not** mean the provider lacks that data. `unavailable` is an observed 404, while `blocked` is an access failure. Empty source data remain explicitly empty. A match row can exist from the cached identity even with no successful live components.

Counts include website requests for unrelated data and any reported browser-cache hits. Compare successful runs with equivalent features before concluding that this route is more efficient. The saved initial run predates the final page-identity reporting guards; its screenshot and UI snapshot record the wrong fixture.

In [5]:
report = run_report
if report is None and (SAVED_RUN / "report.json").is_file():
    report = json.loads((SAVED_RUN / "report.json").read_text(encoding="utf-8"))
    print("Saved diagnostic:", SAVED_RUN)
if report is None:
    print("No captured experiment is available yet.")
else:
    keys = ["event_id", "complete", "stop_reason", "event_metadata_source",
            "observed_api_requests", "target_api_requests", "other_api_requests",
            "duplicate_api_requests", "browser_cache_responses", "http_status_counts",
            "observed_api_entries", "duplicate_observations_skipped", "requests_sent_by_importer",
            "page_event_id", "page_identity_matches", "directory"]
    display(pd.DataFrame([{"metric":key, "value":str(report[key])} for key in keys if key in report]))
    display(pd.DataFrame(report["coverage"])[["component", "team_id", "status", "path"]])
    print("Parquet rows:", report.get("table_rows", {}))

Saved diagnostic: C:\Users\luisi\Documents\Programming\Python\xDiyo\data\website_capture_v3\20260912T011212Z-160350af


,metric,value
0,event_id,12499375
1,complete,False
2,stop_reason,None
3,event_metadata_source,reference_from_existing_cache
4,observed_api_requests,56
5,target_api_requests,1
6,other_api_requests,55
7,duplicate_api_requests,8
8,http_status_counts,{'403': 56}
9,directory,C:\Users\luisi\Documents\Programming\Python\xD...


,component,team_id,status,path
0,pregame,NaN,not_observed,/event/12499375/pregame-form
1,statistics,NaN,not_observed,/event/12499375/statistics
2,shotmap,NaN,not_observed,/event/12499375/shotmap
3,heatmap,2527.0,not_observed,/event/12499375/heatmap/2527
4,heatmap,2681.0,not_observed,/event/12499375/heatmap/2681
5,lineups,NaN,not_observed,/event/12499375/lineups
6,incidents,NaN,not_observed,/event/12499375/incidents
7,event,NaN,blocked,/event/12499375


Parquet rows: {'coverage': 8, 'matches': 1, 'timing': 2}


## 6. Inspect prepared data locally

Only existing local Parquet files are read here. Raw-response hashes and observation timestamps allow comparisons later. Missing xG remains null, absent heatmaps are not fabricated, and goal sequences appear only when the provider includes them. Match/half timing retains the existing parser's evidence and uncertainty.

This one-match prototype does not produce normalized seasonal standings, promotion/relegation flags, or legacy-named season files. The original seasonal pipeline remains available separately.

In [6]:
tables = {}
if report and report.get("manifest") and Path(report["manifest"]).is_file():
    tables = read_tables(report["manifest"])
    for name in ("matches", "shots", "lineup_players", "availability", "heatmap_points", "goal_actions"):
        if name in tables:
            print(f"{name}: {len(tables[name])} rows")
            preview = pd.DataFrame(tables[name]).drop(columns=["source_item_json"], errors="ignore")
            display(preview.head(3).iloc[:, :10])
else:
    print("No prepared export to inspect.")

matches: 1 rows


,event_id,competition_id,tournament_id,season_id,status_code,round,home_id,away_id,custom_id,season_year
0,12499375,35,42,63516,100,1,2527,2681,CabsGdb,24/25


## Validation and next decision

The notebook is executed top-to-bottom with live collection disabled. Offline browser-event, HAR, and extension-adapter tests exercise capture, isolation, failure handling, xG/lineup/goal parsing, and Parquet export; synthetic heatmaps test transformation only. A separate notebook execution supplies a fixture HAR to test the complete import path.

The first Selenium diagnostic captured no successful payloads. The existing Chrome profile did load the historical match, but the new extension's browser integration and a real HAR export still need live validation. These are separate from the passing offline tests. Do not scale this experiment to seasons before checking an actual exported match's component coverage.